# 🌳 Module 4: Decision Trees & Random Forests
## Theory + Practical

---

## 📚 THEORY

### What is a Decision Tree?
A flowchart-like model that splits data using feature tests.

```
[Petal Length > 2.5?]
  YES → [Petal Width > 1.75?]   NO → Setosa
            YES → Virginica
            NO  → Versicolor
```

### How Does it Split? — Gini Impurity

```
Gini(S) = 1 - Σ pᵢ²

Gini = 0    → Pure node (all same class) ← BEST
Gini = 0.5  → Maximum impurity
```

### Information Gain
```
IG = Entropy(parent) - weighted_avg(Entropy(children))
→ Always pick the split with HIGHEST information gain
```

### Stopping Overfitting

| Parameter | Effect |
|-----------|--------|
| `max_depth` | Limits tree depth |
| `min_samples_split` | Min samples to allow a split |
| `min_samples_leaf` | Min samples required at a leaf |

---

### Random Forest 🌲🌲🌲

Single trees overfit. Fix: build MANY trees on random subsets.

```
1. Bootstrap sample (random rows with replacement)
2. At each split: use random subset of features
3. Final prediction = majority vote (classification) or mean (regression)
```

**Why it works:** Many diverse, weakly correlated trees cancel each other's errors.

---

## 🔬 PRACTICAL

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_iris, load_breast_cancer
import warnings; warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

### Practical 1: Gini Impurity from Scratch

In [ ]:
def gini(y):
    if len(y) == 0: return 0
    _, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1 - np.sum(probs ** 2)

# Examples
print('Pure node (all same class):    Gini =', gini([0,0,0,0]))
print('50/50 split (max impurity):    Gini =', gini([0,0,1,1]))
print('Mixed 3:1:                     Gini =', gini([0,0,0,1]))

# Visualize Gini vs class proportion
p = np.linspace(0, 1, 200)
gini_vals = 1 - (p**2 + (1-p)**2)
plt.figure(figsize=(7, 4))
plt.plot(p, gini_vals, linewidth=2.5, color='steelblue')
plt.axvline(0.5, color='red', linestyle='--', label='Max impurity at p=0.5')
plt.title('Gini Impurity vs Class Proportion')
plt.xlabel('Proportion of Class 1')
plt.ylabel('Gini Impurity')
plt.legend()
plt.tight_layout()
plt.show()

### Practical 2: Decision Tree — Depth & Overfitting

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

depths = [1, 2, 3, 5, 10, None]
train_scores, test_scores = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, dt.predict(X_train)))
    test_scores.append(accuracy_score(y_test, dt.predict(X_test)))

labels = [str(d) if d else 'None' for d in depths]
plt.figure(figsize=(8, 4))
plt.plot(labels, train_scores, 'bo-', label='Train')
plt.plot(labels, test_scores, 'ro-', label='Test')
plt.title('Decision Tree: Depth vs Accuracy')
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the tree (depth=3 is readable)
best_dt = DecisionTreeClassifier(max_depth=3, random_state=42)
best_dt.fit(X_train, y_train)

plt.figure(figsize=(14, 5))
plot_tree(best_dt, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree (max_depth=3)')
plt.tight_layout()
plt.show()

print(export_text(best_dt, feature_names=list(iris.feature_names)))

### Practical 3: Random Forest vs Single Tree

In [ ]:
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Single tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_score = accuracy_score(y_test, dt.predict(X_test))

# Random Forest with growing n_estimators
n_list = [1, 5, 10, 25, 50, 100, 200]
rf_scores = []
for n in n_list:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_scores.append(accuracy_score(y_test, rf.predict(X_test)))

plt.figure(figsize=(8, 4))
plt.semilogx(n_list, rf_scores, 'go-', linewidth=2, markersize=7, label='Random Forest')
plt.axhline(dt_score, color='red', linestyle='--', label=f'Single Tree ({dt_score:.3f})')
plt.title('More Trees → Better Performance')
plt.xlabel('n_estimators (log scale)')
plt.ylabel('Test Accuracy')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Single Decision Tree: {dt_score:.4f}')
print(f'Random Forest (200):  {rf_scores[-1]:.4f}')

### Practical 4: Feature Importance

In [ ]:
import pandas as pd

rf_final = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_final.fit(X_train, y_train)

importance = pd.Series(rf_final.feature_importances_, index=cancer.feature_names)
top10 = importance.nlargest(10).sort_values()

plt.figure(figsize=(8, 4))
top10.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

### Practical 5: Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid,
                    cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

print(f'Best Params: {grid.best_params_}')
print(f'Best CV Score: {grid.best_score_:.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, grid.best_estimator_.predict(X_test)):.4f}')

### 🏋️ Try It Yourself!

In [ ]:
# ✏️ YOUR TURN: Change max_depth and see train vs test accuracy
max_depth = 5   # Try: 1, 3, 5, 10, None

my_dt = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
my_dt.fit(X_train, y_train)
tr = accuracy_score(y_train, my_dt.predict(X_train))
te = accuracy_score(y_test, my_dt.predict(X_test))
print(f'max_depth={max_depth}: Train={tr:.4f}, Test={te:.4f}, Gap={tr-te:.4f}')
if tr - te > 0.05:
    print('⚠️  Large gap → overfitting! Try smaller max_depth')
else:
    print('✅ Small gap → good generalization')

## 🎓 Summary

| Concept | Takeaway |
|---------|----------|
| Gini Impurity | Measures node purity (0 = pure) |
| Information Gain | Pick split with highest gain |
| max_depth | Controls overfitting |
| Random Forest | Ensemble of trees → reduces variance |
| Feature Importance | Average impurity reduction per feature |
| GridSearchCV | Find best hyperparameters with CV |

➡️ **Next: Support Vector Machines (SVM)**